In [1]:
import pandas as pd
import numpy as np

import geopandas as gpd
import folium
from folium import plugins
from folium.features import GeoJsonTooltip

import warnings
warnings.filterwarnings('ignore')

In [2]:
import funciones_kwichon as f

### Tabla de migraciones del 2006 al 2021

In [3]:
migraciones = pd.read_csv('migraciones_2006_2021.csv', index_col=0,low_memory=False)
migraciones

,SEXO,PROVNAC,EDAD,PROVALTA,MUNIALTA,ANOVAR,PROVBAJA,MUNIBAJA,TAMUALTA,TAMUBAJA,CODMUNIALTA,CODMUNIBAJA
0,1,1,2,48,20,2006,1,0,6,1,48020,1000
1,1,1,53,1,0,2006,1,59,1,6,1000,1059
2,1,1,0,25,0,2006,25,203,1,2,25000,25203
3,1,1,62,66,666,2006,1,59,,6,66666,1059
4,1,1,65,28,0,2006,28,79,1,6,28000,28079
...,...,...,...,...,...,...,...,...,...,...,...,...
40996119,6,66,77,66,666,2021,52,1,,4,66666,52001
40996120,6,66,21,66,666,2021,52,1,,4,66666,52001
40996121,1,66,32,66,666,2021,52,1,,4,66666,52001
40996122,1,66,43,66,666,2021,52,1,,4,66666,52001


In [4]:
# Guardo en un nuevo dataframe la tabla excluyendo las filas donde no hay datos de tamualta y tamubaja 

In [5]:
null_ta = migraciones['TAMUBAJA'].value_counts().index[0]
null_ta

' '

In [6]:
solo_migr = migraciones[~((migraciones['TAMUALTA']== null_ta) | (migraciones['TAMUBAJA']== null_ta))]

In [7]:
solo_migr

,SEXO,PROVNAC,EDAD,PROVALTA,MUNIALTA,ANOVAR,PROVBAJA,MUNIBAJA,TAMUALTA,TAMUBAJA,CODMUNIALTA,CODMUNIBAJA
0,1,1,2,48,20,2006,1,0,6,1,48020,1000
1,1,1,53,1,0,2006,1,59,1,6,1000,1059
2,1,1,0,25,0,2006,25,203,1,2,25000,25203
4,1,1,65,28,0,2006,28,79,1,6,28000,28079
5,6,1,3,1,0,2006,1,59,1,6,1000,1059
...,...,...,...,...,...,...,...,...,...,...,...,...
39922030,1,66,26,50,297,2021,52,1,6,4,50297,52001
39922031,1,66,25,50,297,2021,52,1,6,4,50297,52001
39922032,1,66,45,50,297,2021,52,1,6,4,50297,52001
39922033,1,66,31,51,1,2021,52,1,4,4,51001,52001


In [8]:
#solo_migr= solo_migr.copy()
solo_migr[['TAMUALTA','TAMUBAJA']]= solo_migr[['TAMUALTA','TAMUBAJA']].copy().applymap(int)

### Tabla de población por provincia y por año

In [9]:
df_pob = pd.read_csv('./datos/poblacion_provincia_sexo.csv', sep=';')
df_pob

,Provincias,Sexo,Periodo,Total
0,Total,Total,2021,47.385.107
1,Total,Total,2020,47.450.795
2,Total,Total,2019,47.026.208
3,Total,Total,2018,46.722.980
4,Total,Total,2017,46.572.132
...,...,...,...,...
4129,52 Melilla,Mujeres,2000,32.287
4130,52 Melilla,Mujeres,1999,27.977
4131,52 Melilla,Mujeres,1998,29.539
4132,52 Melilla,Mujeres,1997,NaN


In [10]:
# Se prepara el dataframe con la población de cada provincia por año.

# Se eliminan las filas separadas por 'Sexo'
df_pob= df_pob[~((df_pob['Sexo']=='Mujeres')|(df_pob['Sexo']=='Hombres'))]
# Se eliminan las filas donde se indica Total en la columna 'Provincias'
df_pob = df_pob[~(df_pob['Provincias']=='Total')]
# Se elimina la columna Sexo
df_pob.drop(columns='Sexo', inplace=True)
# Se resetea el índice de la tabla
df_pob.reset_index(drop=True, inplace=True)
df_pob

,Provincias,Periodo,Total
0,02 Albacete,2021,386.464
1,02 Albacete,2020,388.270
2,02 Albacete,2019,388.167
3,02 Albacete,2018,388.786
4,02 Albacete,2017,390.032
...,...,...,...
1347,52 Melilla,2000,66.263
1348,52 Melilla,1999,56.929
1349,52 Melilla,1998,60.108
1350,52 Melilla,1997,NaN


In [11]:
# Se crea una columna nueva para separar el código de provincia del nombre de la provincia
df_pob['Cod_prov']= [int(x.split(' ')[0]) for x in df_pob['Provincias']]
df_pob['Provincias'] = df_pob['Provincias'].apply(lambda x: x.split(' ')[1])

In [12]:
df_pob

,Provincias,Periodo,Total,Cod_prov
0,Albacete,2021,386.464,2
1,Albacete,2020,388.270,2
2,Albacete,2019,388.167,2
3,Albacete,2018,388.786,2
4,Albacete,2017,390.032,2
...,...,...,...,...
1347,Melilla,2000,66.263,52
1348,Melilla,1999,56.929,52
1349,Melilla,1998,60.108,52
1350,Melilla,1997,NaN,52


In [13]:
# Se ordenan las columnas
df_pob = df_pob[['Periodo','Cod_prov', 'Provincias', 'Total']]

In [14]:
# Se filtra por los años del estudio
years = [y for y in range(2006, 2022)]
df_pob = df_pob[df_pob['Periodo'].isin(years)]
df_pob

,Periodo,Cod_prov,Provincias,Total
0,2021,2,Albacete,386.464
1,2020,2,Albacete,388.270
2,2019,2,Albacete,388.167
3,2018,2,Albacete,388.786
4,2017,2,Albacete,390.032
...,...,...,...,...
1337,2010,52,Melilla,76.034
1338,2009,52,Melilla,73.460
1339,2008,52,Melilla,71.448
1340,2007,52,Melilla,69.440


In [15]:
# Se ordenapor años y código de provincia
df_pob.sort_values(by=['Periodo','Cod_prov'], inplace=True, ignore_index=True)
df_pob

,Periodo,Cod_prov,Provincias,Total
0,2006,1,Araba/Álava,301.926
1,2006,2,Albacete,387.658
2,2006,3,Alicante/Alacant,1.783.555
3,2006,4,Almería,635.850
4,2006,5,Ávila,167.818
...,...,...,...,...
827,2021,48,Bizkaia,1.154.334
828,2021,49,Zamora,168.725
829,2021,50,Zaragoza,967.452
830,2021,51,Ceuta,83.517


In [16]:
df_pob.columns = ['Periodo', 'Cod_prov', 'Provincias', 'Poblacion']
df_pob

,Periodo,Cod_prov,Provincias,Poblacion
0,2006,1,Araba/Álava,301.926
1,2006,2,Albacete,387.658
2,2006,3,Alicante/Alacant,1.783.555
3,2006,4,Almería,635.850
4,2006,5,Ávila,167.818
...,...,...,...,...
827,2021,48,Bizkaia,1.154.334
828,2021,49,Zamora,168.725
829,2021,50,Zaragoza,967.452
830,2021,51,Ceuta,83.517


In [17]:
df_pob.dtypes

Periodo        int64
Cod_prov       int64
Provincias    object
Poblacion     object
dtype: object

In [18]:
# Se cambia a int la columna Poblacion. Antes se ha de eliminar el . de los valores para poder convertirlos en int

In [19]:
df_pob['Poblacion']= df_pob['Poblacion'].apply(lambda x: x.replace('.', ''))

In [20]:
df_pob

,Periodo,Cod_prov,Provincias,Poblacion
0,2006,1,Araba/Álava,301926
1,2006,2,Albacete,387658
2,2006,3,Alicante/Alacant,1783555
3,2006,4,Almería,635850
4,2006,5,Ávila,167818
...,...,...,...,...
827,2021,48,Bizkaia,1154334
828,2021,49,Zamora,168725
829,2021,50,Zaragoza,967452
830,2021,51,Ceuta,83517


In [21]:
df_pob['Poblacion']= df_pob['Poblacion'].apply(int)

In [22]:
df_pob.dtypes

Periodo        int64
Cod_prov       int64
Provincias    object
Poblacion      int64
dtype: object

# Mapa

### Provincias

In [23]:
df_prov = gpd.read_file('provinces.geojson')
df_prov.head()

,id,name,geometry
0,25,Lleida,"MULTIPOLYGON (((0.38507 41.27846, 0.36483 41.3..."
1,24,León,"POLYGON ((-5.00157 42.28985, -5.03756 42.29792..."
2,23,Jaén,"POLYGON ((-4.00071 37.40257, -4.01646 37.41711..."
3,22,Huesca,"POLYGON ((0.33559 41.40771, 0.31535 41.41741, ..."
4,21,Huelva,"POLYGON ((-6.93132 38.20877, -6.90208 38.20231..."


### Preparación de tabla para mostrar mapas de saldos de migraciones

In [24]:
df_m= solo_migr

In [25]:
# Elimino las filas de los movimientos desde o hacia el extranjero

In [26]:
extranjero = ((df_m['PROVALTA']==66) | (df_m['PROVALTA']==66))
df_m = df_m[~extranjero]
df_m

,SEXO,PROVNAC,EDAD,PROVALTA,MUNIALTA,ANOVAR,PROVBAJA,MUNIBAJA,TAMUALTA,TAMUBAJA,CODMUNIALTA,CODMUNIBAJA
0,1,1,2,48,20,2006,1,0,6,1,48020,1000
1,1,1,53,1,0,2006,1,59,1,6,1000,1059
2,1,1,0,25,0,2006,25,203,1,2,25000,25203
4,1,1,65,28,0,2006,28,79,1,6,28000,28079
5,6,1,3,1,0,2006,1,59,1,6,1000,1059
...,...,...,...,...,...,...,...,...,...,...,...,...
39922030,1,66,26,50,297,2021,52,1,6,4,50297,52001
39922031,1,66,25,50,297,2021,52,1,6,4,50297,52001
39922032,1,66,45,50,297,2021,52,1,6,4,50297,52001
39922033,1,66,31,51,1,2021,52,1,4,4,51001,52001


### Mapa de migraciones por PROVINCIA

In [27]:
# año inicio
start_y = 2020
# año fin
end_y = 2021
year = [x for x in range(start_y, end_y+1)]

In [28]:
df_m_prov = f.tabla_saldo_prov (df_m, year, df_prov, df_pob)
df_m_prov

,id,name,geometry,PROVSALDO,ALTAS,BAJAS,SALDO,Poblacion
0,25,Lleida,"MULTIPOLYGON (((0.38507 41.27846, 0.36483 41.3...",25,33826,30636,3190,439122.0
1,24,León,"POLYGON ((-5.00157 42.28985, -5.03756 42.29792...",24,31714,31309,405,454072.5
2,23,Jaén,"POLYGON ((-4.00071 37.40257, -4.01646 37.41711...",23,24275,27694,-3419,629285.5
3,22,Huesca,"POLYGON ((0.33559 41.40771, 0.31535 41.41741, ...",22,19015,16137,2878,223475.5
4,21,Huelva,"POLYGON ((-6.93132 38.20877, -6.90208 38.20231...",21,28936,27062,1874,525056.5
5,20,Gipuzkoa,"MULTIPOLYGON (((-2.54553 43.08797, -2.50505 43...",20,43792,44247,-455,726577.0
6,19,Guadalajara,"MULTIPOLYGON (((-1.80557 40.39795, -1.83256 40...",19,33025,27108,5917,263791.5
7,18,Granada,"MULTIPOLYGON (((-3.77805 36.73855, -3.78255 36...",18,74783,72998,1785,920253.0
8,17,Girona,"MULTIPOLYGON (((2.01794 42.10890, 2.00219 42.1...",17,74084,66186,7898,784192.0
9,16,Cuenca,"POLYGON ((-2.74120 39.31871, -2.77269 39.36879...",16,13986,12907,1079,195827.5


In [29]:
mapa = f.plot_map_saldo_prov(df_prov,df_m_prov,year)

In [30]:
mapa

In [31]:
# save map
#outfp = "results/choropleth_map.html"
#m.save(outfp)